# 01 — Data Understanding

Explore the raw e-commerce fraud data before cleaning or modeling.

Objectives:
- Inspect dataset dimensions, columns, and data types.
- Check missing values and duplicate rows.
- Review the target distribution and descriptive statistics.

Inputs: `data/raw/Fraud_Data.csv` and `data/raw/IpAddress_to_Country.csv`. The target is `class`: `1` = fraud, `0` = legitimate.

Execution requires Python 3, pandas, NumPy, and IPython; nbclient and ipykernel are used for automated execution.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find this repository, rather than a CSV in a parent project folder.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "environment.yml").is_file() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch this notebook from within the team repository.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
required_files = ["Fraud_Data.csv", "IpAddress_to_Country.csv"]
missing_files = [name for name in required_files if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing raw files in {RAW_DIR}: {', '.join(missing_files)}. "
        "Download the Kaggle CSVs into data/raw/ before running this notebook."
    )

fraud = pd.read_csv(RAW_DIR / "Fraud_Data.csv")
ip_countries = pd.read_csv(RAW_DIR / "IpAddress_to_Country.csv")


## Dataset overview


In [2]:
for name, frame in {"Purchases": fraud, "IP country lookup": ip_countries}.items():
    print(f"{name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    display(frame.head())
    frame.info()


Purchases: 151,112 rows × 11 columns


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  object 
 2   purchase_time   151112 non-null  object 
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  object 
 5   source          151112 non-null  object 
 6   browser         151112 non-null  object 
 7   sex             151112 non-null  object 
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), object(6)
memory usage: 12.7+ MB
IP country lookup: 138,846 rows × 3 columns


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138846 entries, 0 to 138845
Data columns (total 3 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   lower_bound_ip_address  138846 non-null  float64
 1   upper_bound_ip_address  138846 non-null  int64  
 2   country                 138846 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 3.2+ MB


## Initial quality checks

These checks describe the raw files; they do not modify the data.


In [3]:
for name, frame in {"Purchases": fraud, "IP country lookup": ip_countries}.items():
    print(name)
    display(pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_count": frame.isna().sum(),
        "missing_pct": frame.isna().mean().mul(100),
    }))
    print(f"Duplicate rows: {frame.duplicated().sum():,}")


Purchases


,dtype,missing_count,missing_pct
user_id,int64,0,0.0
signup_time,object,0,0.0
purchase_time,object,0,0.0
purchase_value,int64,0,0.0
device_id,object,0,0.0
source,object,0,0.0
browser,object,0,0.0
sex,object,0,0.0
age,int64,0,0.0
ip_address,float64,0,0.0


Duplicate rows: 0
IP country lookup


,dtype,missing_count,missing_pct
lower_bound_ip_address,float64,0,0.0
upper_bound_ip_address,int64,0,0.0
country,object,0,0.0


Duplicate rows: 0


## Purchase statistics and target distribution


In [4]:
display(fraud.describe(include="all").T)

target_counts = fraud["class"].value_counts(dropna=False).sort_index()
display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts.div(len(fraud)).mul(100),
}))


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,151112.0,NaN,NaN,NaN,200171.04097,115369.285024,2.0,100642.5,199958.0,300054.0,400000.0
signup_time,151112,151112,2015-06-10 07:02:20,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchase_time,151112,150679,2015-07-17 23:22:55,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchase_value,151112.0,NaN,NaN,NaN,36.935372,18.322762,9.0,22.0,35.0,49.0,154.0
device_id,151112,137956,NGQCKIADMZORL,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,151112,3,SEO,60615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
browser,151112,5,Chrome,61432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,151112,2,M,88293,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,151112.0,NaN,NaN,NaN,33.140704,8.617733,18.0,27.0,33.0,39.0,76.0
ip_address,151112.0,NaN,NaN,NaN,2152145330.958826,1248497030.100426,52093.496895,1085933882.53156,2154770162.40662,3243257679.722522,4294850499.67884


,count,percentage
class,,
0,136961,90.635423
1,14151,9.364577


## Timestamp validity and purchase ordering

Parse the documented timestamp format into separate variables, leaving the raw columns intact. The source timestamps contain no timezone information.


In [5]:
timestamps = fraud[["signup_time", "purchase_time"]].apply(
    lambda column: pd.to_datetime(column, format="%Y-%m-%d %H:%M:%S", errors="coerce")
)
display(pd.DataFrame({
    "invalid_or_missing": timestamps.isna().sum(),
    "earliest": timestamps.min(),
    "latest": timestamps.max(),
}))
lag_seconds = (timestamps["purchase_time"] - timestamps["signup_time"]).dt.total_seconds()
display(pd.Series({
    "purchase_before_signup": int(lag_seconds.lt(0).sum()),
    "purchase_at_signup": int(lag_seconds.eq(0).sum()),
    "purchase_one_second_after_signup": int(lag_seconds.eq(1).sum()),
    "minimum_lag_seconds": lag_seconds.min(),
    "median_lag_days": lag_seconds.median() / 86400,
    "maximum_lag_days": lag_seconds.max() / 86400,
}, name="timestamp_checks"))
display(pd.crosstab(lag_seconds.eq(1).rename("one_second_lag"), fraud["class"]))


,invalid_or_missing,earliest,latest
signup_time,0,2015-01-01 00:00:42,2015-08-18 04:40:29
purchase_time,0,2015-01-01 00:00:44,2015-12-16 02:56:05


purchase_before_signup                 0.000000
purchase_at_signup                     0.000000
purchase_one_second_after_signup    7600.000000
minimum_lag_seconds                    1.000000
median_lag_days                       57.017888
maximum_lag_days                     119.999676
Name: timestamp_checks, dtype: float64

class,0,1
one_second_lag,,
False,136961,6551
True,0,7600


## Numeric ranges, categories, and repeated identifiers

Repeated devices or IPs may be meaningful fraud signals; they are distinct from duplicate transaction rows. Extreme numeric values are flagged for review rather than automatically removed.


In [6]:
import numpy as np

numeric = fraud.select_dtypes(include="number")
display(pd.DataFrame({
    "minimum": numeric.min(),
    "maximum": numeric.max(),
    "non_finite": (~np.isfinite(numeric)).sum(),
    "fractional_values": numeric.mod(1).ne(0).sum(),
}))
display(pd.Series({
    "non_positive_purchase_value": int(fraud["purchase_value"].le(0).sum()),
    "non_positive_age": int(fraud["age"].le(0).sum()),
    "invalid_target": int((~fraud["class"].isin([0, 1])).sum()),
    "ip_outside_ipv4_numeric_bounds": int((~fraud["ip_address"].between(0, 2**32 - 1)).sum()),
}, name="numeric_checks"))
for column in ["source", "browser", "sex"]:
    print(column)
    display(fraud[column].value_counts(dropna=False))

category_checks = []
for name, frame, columns in [
    ("Purchases", fraud, ["source", "browser", "sex", "device_id"]),
    ("IP lookup", ip_countries, ["country"]),
]:
    for column in columns:
        values = frame[column].astype("string")
        stripped = values.str.strip()
        category_checks.append({
            "dataset": name, "column": column,
            "blank_strings": int(stripped.eq("").sum()),
            "surrounding_whitespace": int(values.ne(stripped).sum()),
            "case_or_whitespace_variants": int(values.nunique() - stripped.str.casefold().nunique()),
        })
display(pd.DataFrame(category_checks))

display(pd.DataFrame({
    "unique_values": fraud[["user_id", "device_id", "ip_address"]].nunique(),
    "repeats_beyond_first": [fraud[c].duplicated().sum() for c in ["user_id", "device_id", "ip_address"]],
    "maximum_rows_per_value": [fraud[c].value_counts().max() for c in ["user_id", "device_id", "ip_address"]],
}))
for column in ["purchase_value", "age"]:
    q1, q3 = fraud[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    outside = ~fraud[column].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f"{column}: {int(outside.sum()):,} rows outside 1.5 × IQR fences "
          f"[{q1 - 1.5 * iqr:g}, {q3 + 1.5 * iqr:g}]")


,minimum,maximum,non_finite,fractional_values
user_id,2.000000,4.000000e+05,0,0
purchase_value,9.000000,1.540000e+02,0,0
age,18.000000,7.600000e+01,0,0
ip_address,52093.496895,4.294850e+09,0,151111
class,0.000000,1.000000e+00,0,0


non_positive_purchase_value       0
non_positive_age                  0
invalid_target                    0
ip_outside_ipv4_numeric_bounds    0
Name: numeric_checks, dtype: int64

source


source
SEO       60615
Ads       59881
Direct    30616
Name: count, dtype: int64

browser


...

sex


sex
M    88293
F    62819
Name: count, dtype: int64

,dataset,column,blank_strings,surrounding_whitespace,case_or_whitespace_variants
0,Purchases,source,0,0,0
1,Purchases,browser,0,0,0
2,Purchases,sex,0,0,0
3,Purchases,device_id,0,0,0
4,IP lookup,country,0,0,0


,unique_values,repeats_beyond_first,maximum_rows_per_value
user_id,151112,0,1
device_id,137956,13156,20
ip_address,143512,7600,20


purchase_value: 1,067 rows outside 1.5 × IQR fences [-18.5, 89.5]
age: 776 rows outside 1.5 × IQR fences [9, 57]


## IP lookup validity and purchase coverage

Treat lookup bounds as inclusive. First audit their order, integer validity, and overlap. Purchase IPs are stored as fractional numbers: report coverage using their original values and compare with flooring as a diagnostic only. Flooring is a candidate conversion that still needs source confirmation.


In [7]:
bounds = ip_countries[["lower_bound_ip_address", "upper_bound_ip_address"]]
lower = bounds["lower_bound_ip_address"]
upper = bounds["upper_bound_ip_address"]
lookup = ip_countries.sort_values("lower_bound_ip_address").reset_index(drop=True)
previous_max_upper = lookup["upper_bound_ip_address"].cummax().shift(1)
overlaps = lookup["lower_bound_ip_address"].le(previous_max_upper)
gaps = lookup["lower_bound_ip_address"].gt(previous_max_upper + 1)
invalid_ranges = (~np.isfinite(bounds)).any(axis=1) | (~bounds.ge(0) | ~bounds.le(2**32 - 1)).any(axis=1) | bounds.mod(1).ne(0).any(axis=1) | lower.gt(upper)
display(pd.Series({
    "lookup_rows": len(lookup),
    "countries": lookup["country"].nunique(),
    "invalid_ranges": int(invalid_ranges.sum()),
    "duplicate_bound_pairs": int(bounds.duplicated().sum()),
    "original_lower_bounds_sorted": lower.is_monotonic_increasing,
    "overlapping_ranges": int(overlaps.sum()),
    "internal_gaps": int(gaps.sum()),
    "smallest_bound": lower.min(),
    "largest_bound": upper.max(),
}, name="lookup_checks"))
assert not invalid_ranges.any(), "Resolve invalid IP ranges before mapping."
assert not overlaps.any(), "Resolve ambiguous IP overlaps before mapping."

def match_country(ip_values):
    # Locate the latest range start at or below each address, then check its end.
    values = np.asarray(ip_values, dtype=float)
    starts = lookup["lower_bound_ip_address"].to_numpy()
    ends = lookup["upper_bound_ip_address"].to_numpy()
    positions = np.searchsorted(starts, values, side="right") - 1
    safe_positions = positions.clip(0)
    matched = (np.isfinite(values) & (positions >= 0)
               & (values <= ends[safe_positions])
               & (values >= 0) & (values <= 2**32 - 1))
    countries = pd.Series(pd.NA, index=fraud.index, dtype="string", name="country")
    countries.loc[matched] = lookup["country"].to_numpy()[positions[matched]]
    return countries

raw_country = match_country(fraud["ip_address"])
floored_country = match_country(np.floor(fraud["ip_address"]))
display(pd.DataFrame({
    "matched_purchases": [raw_country.notna().sum(), floored_country.notna().sum()],
    "unmatched_purchases": [raw_country.isna().sum(), floored_country.isna().sum()],
    "coverage_pct": [raw_country.notna().mean() * 100, floored_country.notna().mean() * 100],
}, index=["Original numeric IP", "Floored IP (diagnostic)"]))
print("Assignments changed by flooring:", int(raw_country.fillna("<unmatched>").ne(floored_country.fillna("<unmatched>")).sum()))
unmatched = raw_country.isna()
display(pd.Series({
    "unmatched_below_lookup_minimum": int((unmatched & fraud["ip_address"].lt(lower.min())).sum()),
    "unmatched_above_lookup_maximum": int((unmatched & fraud["ip_address"].gt(upper.max())).sum()),
    "unmatched_inside_lookup_span": int((unmatched & fraud["ip_address"].between(lower.min(), upper.max())).sum()),
}, name="unmatched_location"))
display(pd.crosstab(raw_country.notna().rename("country_matched"), fraud["class"]))


lookup_rows                         138846
countries                              235
invalid_ranges                           0
duplicate_bound_pairs                    0
original_lower_bounds_sorted          True
overlapping_ranges                       0
internal_gaps                         3370
smallest_bound                  16777216.0
largest_bound                   3758096383
Name: lookup_checks, dtype: object

,matched_purchases,unmatched_purchases,coverage_pct
Original numeric IP,129146,21966,85.463762
Floored IP (diagnostic),129146,21966,85.463762


Assignments changed by flooring: 0


unmatched_below_lookup_minimum      634
unmatched_above_lookup_maximum    19383
unmatched_inside_lookup_span       1949
Name: unmatched_location, dtype: int64

class,0,1
country_matched,,
False,20083,1883
True,116878,12268


## Findings and next steps

Findings from the two local CSV files. Supporting checks and outputs appear above; raw files are unchanged.

- **Dataset size and observed class balance:** Purchases contain **151,112 rows × 11 columns**. There are **136,961 legitimate purchases (90.64%)** and **14,151 fraudulent purchases (9.36%)**. The IP lookup contains **138,846 rows × 3 columns**, covering **235 country labels**. The target is imbalanced; a majority-only prediction would miss every fraud case.

- **Missing values and duplicate records:** Both datasets have **zero missing values and zero duplicate rows**. Audited text columns contain no blank strings or surrounding whitespace. All **151,112 user IDs are unique**. Devices and IPs repeat: **137,956 unique devices** (13,156 repeats beyond the first occurrence) and **143,512 unique IP values** (7,600 repeats beyond the first); both have a maximum of 20 purchases per value. Preserve these distinct transactions and investigate shared identifiers as potential signals.

- **Timestamp validity and signup-to-purchase ordering:** All timestamps parse with `%Y-%m-%d %H:%M:%S`. Signups span **2015-01-01 00:00:42 to 2015-08-18 04:40:29**; purchases span **2015-01-01 00:00:44 to 2015-12-16 02:56:05**. No purchase precedes or equals signup. The delay ranges from **1 second to just under 120 days**, with a median of **57.02 days**. **7,600 purchases occur exactly one second after signup, all labeled fraud**; they account for **53.71% of fraud cases**. Investigate this unusually strong pattern and whether it reflects collection or generation artifacts. Timezones are unspecified.

- **Numeric ranges, category consistency, and unusual values:** Purchase value ranges from **9 to 154**, age from **18 to 76**, and user ID from **2 to 400,000**. Numeric fields are finite; purchase values and ages are positive; target values are only `0` and `1`. Source values are `Ads`, `Direct`, and `SEO`; browsers are `Chrome`, `FireFox`, `IE`, `Opera`, and `Safari`; sex values are `F` and `M`. Audited categorical fields show no variants caused only by case or whitespace. The 1.5 × IQR rule flags **1,067 purchase values above 89.5** and **776 ages above 57**; these are review flags, not established errors. Purchase IPs span approximately **52,093.4969 to 4,294,850,499.6788**, within numeric IPv4 bounds, but **151,111 of 151,112 values are fractional**. Their encoding needs confirmation before conversion to standard IP strings or API use. Purchase currency and units also need confirmation.

- **IP lookup range validity, overlap, and purchase coverage:** Bounds are integer-valued, sorted by lower bound, within IPv4 numeric bounds, and correctly ordered (`lower <= upper`), with **zero duplicate bound pairs and zero overlapping ranges**. The lookup spans **16,777,216 to 3,758,096,383** and has **3,370 internal gaps**. Inclusive range matching covers **129,146 purchases (85.46%)** and leaves **21,966 (14.54%) unmatched**: **634 below** the lookup minimum, **19,383 above** its maximum, and **1,949 within gaps in the lookup span**. Flooring purchase IP values changes **zero country assignments** in this dataset, but does not establish the intended encoding. Missing country matches therefore represent lookup coverage limitations rather than missing source IPs.

- **Cleaning decisions and questions for further investigation:** Retain the original CSVs and all transaction rows; no missing-value imputation or row deduplication is indicated by these checks. Parse timestamps into separate datetime columns, document the unknown timezone, and derive signup-to-purchase delay. Confirm IP encoding before adopting an integer conversion; preserve raw IPs and retain unmatched purchases with an explicit unknown country or match-status flag. Keep the observed category spelling, including `FireFox`, unless a documented mapping is introduced. Keep numeric outliers pending source review. Treat `user_id` as an identifier, and construct device/IP history features using only earlier purchases. For modeling, use class-aware metrics and assess a chronological split with shared-device/IP overlap checks. Investigate label provenance, the one-second fraud pattern, currency/units, and whether historical IP-country coverage is appropriate before deciding on further enrichment or exclusions.
